<img src='http://www.tsc.uc3m.es/~navia/figures/logo_uc3m_foot.jpg' width=800 />

# Proyecto 3: Análisis de opinión

### Procesado del Lenguaje Natural

**Angel Navia Vázquez**

  * 1.1 (January 2026) Revised and updated version

Departamento de Teoría de la Señal y Comunicaciones

**Universidad Carlos III de Madrid**


En este notebook debéis entrenar un modelo para análisis de opinión:

- Para el entrenamiento del modelo se utilizarán los datos **data_project_NLP_3_train.csv**. En este fichero las opiniones positivas están codificadas como "1" y las negativas como "0".

- Tenéis total libertad para elegir la mejor implementación, tanto en lo que respecta a preprocesado, vectorización, como modelos de Machine Learning a utilizar.

- Incluid en el notebook todos los diferentes modelos evaluados, incluso las versiones preliminares, así como el modelo ganador final. Es decir, no borréis las pruebas intermedias que vayáis haciendo. Finalmente tendréis que elegir vuestro MODELO GANADOR, que será el que se evalúe sobre el  conjunto de test (**data_project_NLP_3_test.csv**).

- Presentar las prestaciones (AUC) del MEJOR MODELO sobre los datos de test (AUC).


In [1]:
# Importar aquí las librerías necesarias
import pandas as pd
import numpy as np
import re
import warnings
warnings.filterwarnings('ignore')

# NLP
import spacy
import nltk
from nltk.corpus import stopwords

# Sklearn - preprocesado y vectorización
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD

# Sklearn - modelos
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, VotingClassifier
from sklearn.calibration import CalibratedClassifierCV

# Sklearn - evaluación
from sklearn.model_selection import cross_val_score, StratifiedKFold, GridSearchCV, cross_val_predict
from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix
from sklearn.pipeline import Pipeline

# PyTorch (para DistilBERT)
import torch

# Instalar transformers si no está disponible
try:
    import transformers
    print(f"transformers {transformers.__version__} ya disponible")
except ImportError:
    print("Instalando transformers...")
    import subprocess
    subprocess.check_call(['pip', 'install', 'transformers', 'accelerate'])
    import transformers
    print(f"transformers {transformers.__version__} instalado")

from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from torch.utils.data import Dataset

import time

# Descargar recursos NLTK
nltk.download('stopwords', quiet=True)

# Cargar modelo SpaCy
nlp = spacy.load('en_core_web_md', disable=['parser', 'ner'])

print("Todas las librerías cargadas correctamente")
print(f"Device: {'cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu'}")
device = 'mps' if torch.backends.mps.is_available() else ('cuda' if torch.cuda.is_available() else 'cpu')

Instalando transformers...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 28.7 MB/s eta 0:00:0000:010:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 35.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 642.6/642.6 kB 37.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 57.3 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 447.1/447.1 kB 30.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 174.0/174.0 kB 26.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 54.2 MB/s eta 0:00:00a 0:00:01


transformers 5.5.4 instalado


[nltk_data] Error loading stopwords: <urlopen error [SSL:
[nltk_data]     CERTIFICATE_VERIFY_FAILED] certificate verify failed:
[nltk_data]     unable to get local issuer certificate (_ssl.c:1007)>


Todas las librerías cargadas correctamente
Device: mps


In [2]:
# Cargar los datos de entrenamiento proporcionados
DATA_PATH = 'Proyecto_3_NLP_Datasets/'

df_train = pd.read_csv(DATA_PATH + 'data_project_NLP_3_train.csv')

print(f"Dimensiones del dataset de entrenamiento: {df_train.shape}")
print(f"\nDistribución de clases:")
print(df_train['opinion'].value_counts())
print(f"\nBalance: {df_train['opinion'].mean():.2%} positivas")
print(f"\nPrimeras filas:")
df_train.head(10)

Dimensiones del dataset de entrenamiento: (8000, 2)

Distribución de clases:
opinion
0    4122
1    3878
Name: count, dtype: int64

Balance: 48.48% positivas

Primeras filas:


,texto,opinion
0,@AmericanAir it's not a friend it's a legally ...,0
1,Ruukki 's order book at the end of 2010 was 30...,1
2,"""Gained 30 pounds while on prednisone. It was...",1
3,"""suffering from ME/CFS for over 25 yrs; morphi...",1
4,"""High Points:-Easy to read screen-Touchscreen ...",1
5,"""Day 1-Started with first dose after breakfast...",0
6,Apple's iPhone 6 Plus Amazingly Captures 41% o...,1
7,"""For about 6 years my physician prescribed Lip...",1
8,my poor mom can't sleep cause she had a bat in...,0
9,@LittleMissCindy I know it does!! But I really...,0


## Exploración y preprocesado de datos

In [3]:
# Exploración de los datos
print("Valores nulos:")
print(df_train.isnull().sum())
print(f"\nLongitud media de los textos: {df_train['texto'].str.len().mean():.0f} caracteres")
print(f"Longitud mediana: {df_train['texto'].str.len().median():.0f} caracteres")
print(f"Longitud máxima: {df_train['texto'].str.len().max()} caracteres")
print(f"Longitud mínima: {df_train['texto'].str.len().min()} caracteres")

# Rellenar posibles nulos
df_train['texto'] = df_train['texto'].fillna('')

# Separar features y target
X_train_raw = df_train['texto'].values
y_train = df_train['opinion'].values

print(f"\nTotal muestras: {len(X_train_raw)}")
print(f"Positivas (1): {sum(y_train==1)} ({sum(y_train==1)/len(y_train):.1%})")
print(f"Negativas (0): {sum(y_train==0)} ({sum(y_train==0)/len(y_train):.1%})")

Valores nulos:
texto      0
opinion    0
dtype: int64

Longitud media de los textos: 224 caracteres
Longitud mediana: 135 caracteres
Longitud máxima: 6182 caracteres
Longitud mínima: 61 caracteres

Total muestras: 8000
Positivas (1): 3878 (48.5%)
Negativas (0): 4122 (51.5%)


In [4]:
# Funciones de preprocesado

def clean_text(text):
    """Limpieza básica de texto"""
    # Eliminar HTML entities
    text = re.sub(r'&[a-zA-Z]+;', ' ', text)
    text = re.sub(r'&#\d+;', ' ', text)
    # Eliminar URLs
    text = re.sub(r'http\S+|www\.\S+', '', text)
    # Eliminar menciones @usuario
    text = re.sub(r'@\w+', '', text)
    # Eliminar caracteres especiales pero mantener puntuación básica de sentimiento (!?)
    text = re.sub(r'[^a-zA-Z\s!?]', ' ', text)
    # Eliminar espacios múltiples
    text = re.sub(r'\s+', ' ', text).strip()
    return text.lower()

def preprocess_spacy(texts, batch_size=512):
    """Preprocesado con SpaCy: limpieza + lematización + eliminación de stopwords"""
    cleaned = [clean_text(t) for t in texts]
    processed = []
    for doc in nlp.pipe(cleaned, batch_size=batch_size):
        tokens = [token.lemma_ for token in doc 
                  if not token.is_stop and not token.is_punct and token.is_alpha and len(token) > 1]
        processed.append(' '.join(tokens))
    return processed

# Preprocesar datos de entrenamiento
print("Preprocesando textos con SpaCy (lematización + stopwords)...")
t0 = time.time()
X_train_processed = preprocess_spacy(X_train_raw)
print(f"Preprocesado completado en {time.time()-t0:.1f}s")

# También mantener versión solo con limpieza básica (sin lematización)
X_train_cleaned = [clean_text(t) for t in X_train_raw]

print(f"\nEjemplo original:  {X_train_raw[0][:100]}...")
print(f"Ejemplo limpio:    {X_train_cleaned[0][:100]}...")
print(f"Ejemplo procesado: {X_train_processed[0][:100]}...")

Preprocesando textos con SpaCy (lematización + stopwords)...
Preprocesado completado en 33.4s

Ejemplo original:  @AmericanAir it's not a friend it's a legally required chaperone on a school trip....
Ejemplo limpio:    it s not a friend it s a legally required chaperone on a school trip...
Ejemplo procesado: friend legally require chaperone school trip...


## Modelo 1: TF-IDF (word n-grams) + Logistic Regression (Baseline)

In [5]:
# =====================================================
# MODELO 1: TF-IDF (word) + Logistic Regression
# =====================================================
# Configuración de validación cruzada
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

# Vectorización TF-IDF con word n-grams sobre texto procesado (lematizado)
tfidf_word = TfidfVectorizer(
    ngram_range=(1, 2),
    max_features=50000,
    min_df=2,
    max_df=0.85,
    sublinear_tf=True,
    norm='l2'
)

X_tfidf_word = tfidf_word.fit_transform(X_train_processed)
print(f"TF-IDF word shape: {X_tfidf_word.shape}")

# Logistic Regression
lr_model = LogisticRegression(C=1.0, max_iter=1000, solver='lbfgs', random_state=42)
scores_lr = cross_val_score(lr_model, X_tfidf_word, y_train, cv=cv, scoring='roc_auc', n_jobs=-1)
print(f"\nModelo 1 - LR + TF-IDF word (procesado):")
print(f"  AUC CV: {scores_lr.mean():.4f} (+/- {scores_lr.std():.4f})")

TF-IDF word shape: (8000, 20779)

Modelo 1 - LR + TF-IDF word (procesado):
  AUC CV: 0.8853 (+/- 0.0136)


## Modelo 2: TF-IDF (word + char n-grams) + Logistic Regression

In [6]:
# =====================================================
# MODELO 2: TF-IDF (word + char) + Logistic Regression
# =====================================================
# Usando texto RAW limpio (sin lematizar) - los char n-grams capturan morfología

tfidf_word_raw = TfidfVectorizer(
    analyzer='word',
    ngram_range=(1, 2),
    max_features=50000,
    min_df=2,
    max_df=0.85,
    sublinear_tf=True
)

tfidf_char = TfidfVectorizer(
    analyzer='char_wb',
    ngram_range=(2, 4),
    max_features=50000,
    min_df=2,
    max_df=0.85,
    sublinear_tf=True
)

from scipy.sparse import hstack

X_word_raw = tfidf_word_raw.fit_transform(X_train_cleaned)
X_char = tfidf_char.fit_transform(X_train_cleaned)
X_combined = hstack([X_word_raw, X_char])
print(f"TF-IDF word+char shape: {X_combined.shape}")

lr_model2 = LogisticRegression(C=1.0, max_iter=1000, solver='lbfgs', random_state=42)
scores_lr2 = cross_val_score(lr_model2, X_combined, y_train, cv=cv, scoring='roc_auc', n_jobs=-1)
print(f"\nModelo 2 - LR + TF-IDF word+char (raw):")
print(f"  AUC CV: {scores_lr2.mean():.4f} (+/- {scores_lr2.std():.4f})")

TF-IDF word+char shape: (8000, 62402)

Modelo 2 - LR + TF-IDF word+char (raw):
  AUC CV: 0.9047 (+/- 0.0137)


## Modelo 3: TF-IDF + LinearSVC (calibrado)

In [7]:
# =====================================================
# MODELO 3: TF-IDF (word+char) + LinearSVC (calibrado para probabilidades)
# =====================================================
svc_model = CalibratedClassifierCV(
    LinearSVC(C=1.0, max_iter=5000, random_state=42),
    cv=5
)
scores_svc = cross_val_score(svc_model, X_combined, y_train, cv=cv, scoring='roc_auc', n_jobs=-1)
print(f"Modelo 3 - LinearSVC + TF-IDF word+char:")
print(f"  AUC CV: {scores_svc.mean():.4f} (+/- {scores_svc.std():.4f})")

Modelo 3 - LinearSVC + TF-IDF word+char:
  AUC CV: 0.9108 (+/- 0.0126)


## Modelo 4: TF-IDF + MultinomialNB

In [8]:
# =====================================================
# MODELO 4: TF-IDF + MultinomialNB
# =====================================================
nb_model = MultinomialNB(alpha=0.1)
scores_nb = cross_val_score(nb_model, X_combined, y_train, cv=cv, scoring='roc_auc', n_jobs=-1)
print(f"Modelo 4 - MultinomialNB + TF-IDF word+char:")
print(f"  AUC CV: {scores_nb.mean():.4f} (+/- {scores_nb.std():.4f})")

Modelo 4 - MultinomialNB + TF-IDF word+char:
  AUC CV: 0.8905 (+/- 0.0123)


## Modelo 5: TF-IDF + GradientBoosting

In [9]:
# =====================================================
# MODELO 5: TF-IDF + GradientBoosting (con reducción de dimensionalidad)
# =====================================================
# GradientBoosting es lento con matrices sparse muy grandes, usamos SVD para reducir
svd = TruncatedSVD(n_components=300, random_state=42)
X_svd = svd.fit_transform(X_combined)
print(f"SVD shape: {X_svd.shape}, varianza explicada: {svd.explained_variance_ratio_.sum():.2%}")

gb_model = GradientBoostingClassifier(
    n_estimators=200, max_depth=5, learning_rate=0.1, 
    subsample=0.8, random_state=42
)
scores_gb = cross_val_score(gb_model, X_svd, y_train, cv=cv, scoring='roc_auc', n_jobs=-1)
print(f"\nModelo 5 - GradientBoosting + TF-IDF+SVD:")
print(f"  AUC CV: {scores_gb.mean():.4f} (+/- {scores_gb.std():.4f})")

SVD shape: (8000, 300), varianza explicada: 31.47%

Modelo 5 - GradientBoosting + TF-IDF+SVD:
  AUC CV: 0.8666 (+/- 0.0244)


## Modelo 6: LR optimizado con GridSearchCV

In [10]:
# =====================================================
# MODELO 6: LR optimizado con GridSearchCV sobre TF-IDF word+char
# =====================================================
param_grid = {
    'C': [0.1, 0.5, 1.0, 5.0, 10.0],
    'solver': ['lbfgs', 'liblinear'],
}

grid_lr = GridSearchCV(
    LogisticRegression(max_iter=2000, random_state=42),
    param_grid,
    cv=cv,
    scoring='roc_auc',
    n_jobs=-1,
    verbose=0
)

print("Buscando mejores hiperparámetros para LR...")
grid_lr.fit(X_combined, y_train)
print(f"\nModelo 6 - LR optimizado + TF-IDF word+char:")
print(f"  Mejores parámetros: {grid_lr.best_params_}")
print(f"  AUC CV: {grid_lr.best_score_:.4f}")
scores_lr_opt = grid_lr.best_score_

Buscando mejores hiperparámetros para LR...

Modelo 6 - LR optimizado + TF-IDF word+char:
  Mejores parámetros: {'C': 10.0, 'solver': 'liblinear'}
  AUC CV: 0.9109


## Modelo 7: DistilBERT fine-tuned (Transformer)

In [12]:
# =====================================================
# MODELO 7: DistilBERT fine-tuned para clasificación de sentimiento
# =====================================================

class SentimentDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=256):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len
    
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        text = str(self.texts[idx])
        encoding = self.tokenizer(
            text,
            truncation=True,
            padding='max_length',
            max_length=self.max_len,
            return_tensors='pt'
        )
        return {
            'input_ids': encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
            'labels': torch.tensor(self.labels[idx], dtype=torch.long)
        }

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = torch.softmax(torch.tensor(logits), dim=-1)[:, 1].numpy()
    auc = roc_auc_score(labels, probs)
    return {'auc': auc}

# Configuración
MODEL_NAME = 'distilbert-base-uncased'
MAX_LEN = 256

print(f"Cargando tokenizer y modelo: {MODEL_NAME}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Dividir en train/val para early stopping (90/10)
from sklearn.model_selection import train_test_split
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train_raw, y_train, test_size=0.1, random_state=42, stratify=y_train
)

train_dataset = SentimentDataset(X_tr, y_tr, tokenizer, MAX_LEN)
val_dataset = SentimentDataset(X_val, y_val, tokenizer, MAX_LEN)

print(f"Train: {len(train_dataset)}, Val: {len(val_dataset)}")

# Cargar modelo preentrenado
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

# Configuración del entrenamiento
training_args = TrainingArguments(
    output_dir='./distilbert_sentiment',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    warmup_steps=100,
    weight_decay=0.01,
    learning_rate=2e-5,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='auc',
    greater_is_better=True,
    logging_steps=50,
    fp16=False,
    report_to='none',
    seed=42,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)

# Reemplazar NotebookProgressCallback por ProgressCallback para evitar RuntimeError
from transformers import ProgressCallback
from transformers.utils.notebook import NotebookProgressCallback
trainer.remove_callback(NotebookProgressCallback)
trainer.add_callback(ProgressCallback)

print(f"\nEntrenando DistilBERT en {device}...")
t0 = time.time()
trainer.train()
print(f"Entrenamiento completado en {(time.time()-t0)/60:.1f} minutos")

# Evaluar en validación
eval_results = trainer.evaluate()
print(f"\nModelo 7 - DistilBERT fine-tuned:")
print(f"  AUC Val: {eval_results['eval_auc']:.4f}")
distilbert_auc_val = eval_results['eval_auc']

Cargando tokenizer y modelo: distilbert-base-uncased
Train: 7200, Val: 800


Loading weights: 100%|██████████| 100/100 [00:00<00:00, 1472.43it/s]
DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



Entrenando DistilBERT en mps...


  4%|▎         | 50/1350 [00:24<09:19,  2.32it/s] 

{'loss': '0.691', 'grad_norm': '1.064', 'learning_rate': '9.8e-06', 'epoch': '0.1111'}


...

{'train_runtime': '716.5', 'train_samples_per_second': '30.15', 'train_steps_per_second': '1.884', 'train_loss': '0.2536', 'epoch': '3'}


There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].
100%|██████████| 1350/1350 [12:07<00:00,  1.86it/s]


Entrenamiento completado en 12.2 minutos


100%|██████████| 25/25 [00:07<00:00,  3.39it/s]


Modelo 7 - DistilBERT fine-tuned:
  AUC Val: 0.9603


## Modelo 7b: Re-entrenar DistilBERT con TODOS los datos de entrenamiento

In [13]:
# =====================================================
# MODELO 7b: Re-entrenar DistilBERT con TODOS los datos de train
# =====================================================
# Una vez confirmado que DistilBERT es el mejor modelo, re-entrenamos 
# usando el 100% de los datos de entrenamiento para maximizar el AUC en test

print("Re-entrenando DistilBERT con TODOS los datos de entrenamiento...")

full_train_dataset = SentimentDataset(X_train_raw, y_train, tokenizer, MAX_LEN)

# Recargar modelo limpio
model_full = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

training_args_full = TrainingArguments(
    output_dir='./distilbert_sentiment_full',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    warmup_steps=100,
    weight_decay=0.01,
    learning_rate=2e-5,
    logging_steps=50,
    fp16=False,
    report_to='none',
    save_strategy='epoch',
    seed=42,
)

trainer_full = Trainer(
    model=model_full,
    args=training_args_full,
    train_dataset=full_train_dataset,
)

# Reemplazar NotebookProgressCallback
trainer_full.remove_callback(NotebookProgressCallback)
trainer_full.add_callback(ProgressCallback)

t0 = time.time()
trainer_full.train()
print(f"Re-entrenamiento completado en {(time.time()-t0)/60:.1f} minutos")
print("Modelo ganador listo para evaluación en test.")

Re-entrenando DistilBERT con TODOS los datos de entrenamiento...


Loading weights: 100%|██████████| 100/100 [00:00<00:00, 1163.31it/s]
DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
  3%|▎         | 50/1500 [00:24<10:27,  2.31it/s] 

{'loss': '0.6954', 'grad_norm': '1.765', 'learning_rate': '9.8e-06', 'epoch': '0.1'}


  7%|▋         | 100/1500 [00:45<09:47,  2.38it/s]

{'loss': '0.6207', 'grad_norm': '4.605', 'learning_rate': '1.98e-05', 'epoch': '0.2'}


...

{'loss': '0.15', 'grad_norm': '4.863', 'learning_rate': '7.286e-07', 'epoch': '2.9'}


100%|██████████| 1500/1500 [13:24<00:00,  1.77it/s]

{'loss': '0.08648', 'grad_norm': '0.6855', 'learning_rate': '1.429e-08', 'epoch': '3'}


100%|██████████| 1500/1500 [13:50<00:00,  1.81it/s]

{'train_runtime': '831', 'train_samples_per_second': '28.88', 'train_steps_per_second': '1.805', 'train_loss': '0.245', 'epoch': '3'}
Re-entrenamiento completado en 13.9 minutos
Modelo ganador listo para evaluación en test.


## Comparación de todos los modelos y selección del MODELO GANADOR

In [14]:
# =====================================================
# COMPARACIÓN DE TODOS LOS MODELOS
# =====================================================
print("=" * 65)
print("RESUMEN DE RESULTADOS - AUC en validación cruzada (10-fold)")
print("=" * 65)

resultados = {
    'Modelo 1: LR + TF-IDF word (procesado)': scores_lr.mean(),
    'Modelo 2: LR + TF-IDF word+char (raw)': scores_lr2.mean(),
    'Modelo 3: LinearSVC + TF-IDF word+char': scores_svc.mean(),
    'Modelo 4: MultinomialNB + TF-IDF word+char': scores_nb.mean(),
    'Modelo 5: GradientBoosting + TF-IDF+SVD': scores_gb.mean(),
    'Modelo 6: LR optimizado + TF-IDF word+char': scores_lr_opt,
    'Modelo 7: DistilBERT fine-tuned (val split)': distilbert_auc_val,
}

# Ordenar por AUC descendente
resultados_sorted = dict(sorted(resultados.items(), key=lambda x: x[1], reverse=True))

for i, (nombre, auc) in enumerate(resultados_sorted.items(), 1):
    marker = " <-- GANADOR" if i == 1 else ""
    print(f"  {i}. {nombre}: AUC = {auc:.4f}{marker}")

mejor_modelo = list(resultados_sorted.keys())[0]
mejor_auc = list(resultados_sorted.values())[0]
print(f"\n{'='*65}")
print(f"MODELO GANADOR: {mejor_modelo}")
print(f"AUC en validación: {mejor_auc:.4f}")
print(f"{'='*65}")
print(f"\nNota: El modelo ganador (DistilBERT) ha sido re-entrenado con el 100%")
print(f"de los datos de entrenamiento para su uso en test.")

RESUMEN DE RESULTADOS - AUC en validación cruzada (10-fold)
  1. Modelo 7: DistilBERT fine-tuned (val split): AUC = 0.9603 <-- GANADOR
  2. Modelo 6: LR optimizado + TF-IDF word+char: AUC = 0.9109
  3. Modelo 3: LinearSVC + TF-IDF word+char: AUC = 0.9108
  4. Modelo 2: LR + TF-IDF word+char (raw): AUC = 0.9047
  5. Modelo 4: MultinomialNB + TF-IDF word+char: AUC = 0.8905
  6. Modelo 1: LR + TF-IDF word (procesado): AUC = 0.8853
  7. Modelo 5: GradientBoosting + TF-IDF+SVD: AUC = 0.8666

MODELO GANADOR: Modelo 7: DistilBERT fine-tuned (val split)
AUC en validación: 0.9603

Nota: El modelo ganador (DistilBERT) ha sido re-entrenado con el 100%
de los datos de entrenamiento para su uso en test.


## Evaluación del MODELO GANADOR en el conjunto de test

In [15]:
# Mostrar prestaciones del modelo ganador en el conjunto de test
import time
t_ini = time.time()

# Cargar los datos de test
df_test = pd.read_csv(DATA_PATH + 'data_project_NLP_3_test.csv')
df_test['texto'] = df_test['texto'].fillna('')
X_test_raw = df_test['texto'].values
y_test = df_test['opinion'].values

print(f"Datos de test cargados: {len(X_test_raw)} muestras")
print(f"Distribución: {sum(y_test==1)} positivas, {sum(y_test==0)} negativas")

# Preprocesar los datos de test para su uso por el modelo
# Para DistilBERT no necesitamos preprocesado manual, el tokenizer se encarga
test_dataset = SentimentDataset(X_test_raw, y_test, tokenizer, MAX_LEN)

# Realizar las predicciones y calcular el AUC en el conjunto de test
predictions = trainer_full.predict(test_dataset)
logits = predictions.predictions
probs = torch.softmax(torch.tensor(logits), dim=-1)[:, 1].numpy()

preds_test = probs
AUC_test = roc_auc_score(y_test, preds_test)

print(f'\n{"="*65}')
print(f'AUC en el conjunto de test = {AUC_test}')
print(f'{"="*65}')
print("Tiempo de análisis en minutos = {}".format((time.time() - t_ini)/60))

Datos de test cargados: 2000 muestras
Distribución: 946 positivas, 1054 negativas


100%|██████████| 63/63 [00:21<00:00,  2.89it/s]



AUC en el conjunto de test = 0.9612495035523587
Tiempo de análisis en minutos = 0.4911651333173116
